In [1]:
import os
# Make chdir idempotent so re-running this cell doesn't step above the project root.
if not os.path.exists("data"):
    os.chdir("..")
import sys
sys.path.insert(0, "./src")

import re
import ast
import json
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scripts.train import train_catboost, train_sklearn_model

In [2]:
def parse_mileage(s):
    """MuCars mileage strings:
       '200 000 - 249 999' -> 225000  (midpoint)
       'Plus de 500 000'   -> 500000
       'Moins de 1 000'    -> 500     (midpoint of 0-1000)
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    low = s.lower()
    nums = re.findall(r"\d[\d ]*\d|\d", s)
    nums = [int(n.replace(" ", "")) for n in nums]
    if len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    if len(nums) == 1:
        if "plus" in low:
            return nums[0]
        if "moins" in low:
            return nums[0] / 2
        return nums[0]
    return np.nan


def parse_fiscal_power(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"(\d+)", str(s))
    return float(m.group(1)) if m else np.nan


def preprocess_mucars(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    df = df[df["Price"].notna() & (df["Price"] >= 5_000) & (df["Price"] <= 5_000_000)]

    df["year"] = pd.to_numeric(df["Year"], errors="coerce")
    df.loc[(df["year"] < 1950) | (df["year"] > 2026), "year"] = np.nan
    df["mileage_mid"]  = df["Mileage"].apply(parse_mileage)
    df["fiscal_power"] = df["Fiscal Power"].apply(parse_fiscal_power)
    df["doors"]        = pd.to_numeric(df["Number of Doors"], errors="coerce")

    for col in ["Brand", "Model", "Condition", "Gearbox", "Fuel"]:
        df[col.lower()] = df[col].astype(str).str.lower().str.strip().fillna("unknown")

    df["price"] = df["Price"]
    return df.reset_index(drop=True)


df_raw = pd.read_csv("./data/mucars.csv")
df_clean = preprocess_mucars(df_raw)
print(f"shape: {df_clean.shape}")
print(f"price summary: min={df_clean['price'].min():,.0f}  median={df_clean['price'].median():,.0f}  max={df_clean['price'].max():,.0f}")

shape: (75733, 25)
price summary: min=5,000  median=110,000  max=5,000,000


In [3]:
_NULLISH = {"nan", "none", "other", "unknown", "n/a", "na", "", "unspecified", "not specified", "missing"}
_DESC_COLS = ["Brand", "Model", "Year", "Gearbox"]


def make_desc(row: pd.Series) -> str:
    parts = []
    for col in _DESC_COLS:
        v = row.get(col)
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s.lower() in _NULLISH:
            continue
        if isinstance(v, float) and v == int(v):
            s = str(int(v))
        parts.append(f"{col}: {s}")
    return ", ".join(parts)


_price = pd.to_numeric(df_raw["Price"], errors="coerce")
df_orig = df_raw[_price.notna() & (_price >= 5_000) & (_price <= 5_000_000)].reset_index(drop=True).copy()
df_clean["_desc"] = df_orig.apply(make_desc, axis=1)

conn = sqlite3.connect("./data/mucars_mappings.db")
mappings = {row[0]: json.loads(row[1]) for row in conn.execute("SELECT description, result FROM mappings")}
conn.close()
print(f"Loaded {len(mappings):,} mappings")

mapping_keys = {k for v in mappings.values() for k in v if k != "equipment"}
for key in sorted(mapping_keys):
    df_clean[f"n_{key}"] = df_clean["_desc"].map(lambda d, k=key: (mappings.get(d) or {}).get(k))

n_cols = [c for c in df_clean.columns if c.startswith("n_")]
print(f"Added {len(n_cols)} n_ columns")
matched = df_clean["_desc"].isin(mappings)
print(f"Coverage: {matched.sum():,} / {len(df_clean):,} rows ({matched.mean()*100:.1f}%)")

Loaded 10,207 mappings
Added 25 n_ columns
Coverage: 75,733 / 75,733 rows (100.0%)


In [4]:
EQUIPMENT_ITEMS = [
    "ABS", "Airbags", "ESP", "Rear Camera", "Parking Sensors",
    "Air Conditioning", "Leather Seats", "Sunroof", "Navigation System/GPS",
    "CD/MP3/Bluetooth", "Onboard Computer", "Cruise Control", "Speed Limiter",
    "Electric Windows", "Central Locking", "Alloy Wheels",
]


def parse_equipment_set(s):
    if pd.isna(s):
        return set()
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return {str(x).strip() for x in v}
    except Exception:
        pass
    return set()


def slug(s):
    return "eq_" + s.lower().replace("/", "_").replace(" ", "_")


equip_sets = df_orig["Equipment"].apply(parse_equipment_set).values
for item in EQUIPMENT_ITEMS:
    df_clean[slug(item)] = [int(item in s) for s in equip_sets]

EQ_COLS = [slug(item) for item in EQUIPMENT_ITEMS]
print(f"Added {len(EQ_COLS)} equipment one-hot cols")
print(f"Mean items per car: {df_clean[EQ_COLS].sum(axis=1).mean():.1f}")

Added 16 equipment one-hot cols
Mean items per car: 6.2


In [5]:
# Equipment in BOTH configs (raw and normalized).
RAW_NUM     = ["year", "mileage_mid", "fiscal_power", "doors"] + EQ_COLS
RAW_CAT     = ["brand", "model", "condition", "gearbox", "fuel"]

N_NUM_COLS  = ["n_engine_power", "n_engine_size", "n_cylinders",
               "n_doors", "n_transmission_gears"]
N_CAT_COLS  = ["n_brand", "n_model", "n_body_type", "n_transmission_type",
               "n_transmission_technology", "n_engine_aspiration", "n_injection_type",
               "n_energy_source", "n_fuel_type", "n_propulsion_system", "n_drive_type"]

N_NUM_COLS = [c for c in N_NUM_COLS if c in df_clean.columns]
N_CAT_COLS = [c for c in N_CAT_COLS if c in df_clean.columns]

COMBINED_NUM = RAW_NUM + N_NUM_COLS
COMBINED_CAT = RAW_CAT + N_CAT_COLS

print(f"Total raw      : {len(RAW_NUM) + len(RAW_CAT)} features")
print(f"Total combined : {len(COMBINED_NUM) + len(COMBINED_CAT)} features")

Total raw      : 25 features
Total combined : 41 features


In [6]:
from typing import List, Optional, Tuple


def impute_column(df_train, df_test, column, is_numeric, min_group_size=5, group_by=None):
    if group_by is not None:
        g1, g2 = group_by
    elif column.startswith("n_"):
        g1, g2 = "n_brand", "n_model"
    else:
        g1, g2 = "brand", "model"

    def agg_fn(x):
        if x.count() < min_group_size:
            return None
        return x.median() if is_numeric else (x.mode()[0] if len(x.mode()) > 0 else None)

    bm = df_train.groupby([g1, g2])[column].agg(agg_fn).dropna()
    b  = df_train.groupby(g1)[column].agg(agg_fn).dropna()
    overall = (df_train[column].median() if is_numeric
               else (df_train[column].mode()[0] if not df_train[column].isna().all() else None))

    iv = {(br, mo): v for (br, mo), v in bm.items()}
    for br, v in b.items():
        iv[(br, None)] = v
    iv[("__default__", None)] = overall

    def _fill(row):
        if not pd.isna(row[column]):
            return row[column]
        return iv.get((row[g1], row[g2])) or iv.get((row[g1], None)) or iv[("__default__", None)]

    df_train, df_test = df_train.copy(), df_test.copy()
    if df_train[column].isna().any():
        df_train[column] = df_train.apply(_fill, axis=1)
    if df_test[column].isna().any():
        df_test[column] = df_test.apply(_fill, axis=1)
    return df_train, df_test


def impute_all(df_train, df_test, num_cols, cat_cols, min_group_size=5, group_by=None):
    for col in num_cols:
        df_train, df_test = impute_column(df_train, df_test, col, True, min_group_size, group_by)
    for col in cat_cols:
        df_train, df_test = impute_column(df_train, df_test, col, False, min_group_size, group_by)
    return df_train, df_test


def load_dvm_pairs(path="./data/dvm.csv"):
    df = pd.read_csv(path)
    pairs = set()
    for b, m in zip(df["Automaker"], df["Genmodel"]):
        if pd.isna(b) or pd.isna(m): continue
        pairs.add((str(b).strip().lower(), str(m).strip().lower()))
    return pairs


DVM_PAIRS = load_dvm_pairs()
print(f"DVM-CAR seed catalog: {len(DVM_PAIRS)} (brand, model) pairs")
print("Imputation helpers defined.")

DVM-CAR seed catalog: 1011 (brand, model) pairs
Imputation helpers defined.


In [7]:
config_raw = {
    "num_columns": RAW_NUM, "cat_columns": RAW_CAT,
    "target": "price", "validation": "holdout", "val_size": 0.2,
    "loss_function": "MAE",
}
config_combined = {
    "num_columns": COMBINED_NUM, "cat_columns": COMBINED_CAT,
    "target": "price", "validation": "holdout", "val_size": 0.2,
    "loss_function": "MAE",
}

RANDOM_STATES = list(range(10))
MODEL_KINDS = ["catboost", "xgboost", "lightgbm"]
SLICES = ["all", "merged"]

# (model, cfg) -> per-seed metric arrays for the full test set.
metrics = {(m, c): {"mape": [], "mdape": [], "mae": []}
           for m in MODEL_KINDS for c in ["raw", "norm"]}

# (model, cfg, slice) -> per-seed metric arrays.
slice_metrics = {(m, c, s): {"mape": [], "mdape": [], "mae": [], "n": []}
                 for m in MODEL_KINDS for c in ["raw", "norm"] for s in SLICES}


def slice_stats(y_true, y_pred, mask):
    if mask.sum() == 0:
        return float("nan"), float("nan"), float("nan"), 0
    yt = y_true[mask]; yp = y_pred[mask]
    ape = np.abs((yp - yt) / yt)
    return float(np.mean(ape)), float(np.median(ape)), float(np.mean(np.abs(yp - yt))), int(mask.sum())


def cast_cats(df, cat_cols):
    """sklearn TargetEncoder needs hashable string values; CatBoost handles cats natively."""
    df = df.copy()
    df[cat_cols] = df[cat_cols].fillna("").astype(str).replace("nan", "")
    return df


def fit_predict(model_kind, df_tr, df_te, num_cols, cat_cols, config):
    if model_kind == "catboost":
        model, _ = train_catboost(df_tr, config)
        return np.exp(model.predict(df_te[num_cols + cat_cols]))
    else:
        df_tr_s = cast_cats(df_tr, cat_cols)
        df_te_s = cast_cats(df_te, cat_cols)
        pipe, _ = train_sklearn_model(model_kind, df_tr_s, config)
        return np.exp(pipe.predict(df_te_s[num_cols + cat_cols]))


for rs in RANDOM_STATES:
    df_tr_raw, df_te_raw = train_test_split(df_clean, test_size=0.2, random_state=rs)
    y_te = df_te_raw["price"].values

    # Impute once per seed; reuse across models.
    df_tr_r, df_te_r = impute_all(df_tr_raw, df_te_raw,
                                  ["year", "mileage_mid", "fiscal_power", "doors"],
                                  ["brand", "model", "condition", "gearbox", "fuel"],
                                  group_by=("brand", "model"))
    df_tr_c, df_te_c = impute_all(df_tr_raw, df_te_raw,
                                  ["year", "mileage_mid", "fiscal_power", "doors"],
                                  ["brand", "model", "condition", "gearbox", "fuel"],
                                  group_by=("brand", "model"))
    df_tr_c, df_te_c = impute_all(df_tr_c, df_te_c, N_NUM_COLS, N_CAT_COLS,
                                  group_by=("n_brand", "n_model"))
    for c in EQ_COLS:
        df_tr_r[c] = df_tr_r[c].fillna(0); df_te_r[c] = df_te_r[c].fillna(0)
        df_tr_c[c] = df_tr_c[c].fillna(0); df_te_c[c] = df_te_c[c].fillna(0)

    # Slice masks (computed once, used across all models).
    raw_model = df_te_raw["model"].astype(str).values
    nm = df_te_raw["n_model"]
    nm_str = nm.fillna("").astype(str).str.lower().values
    merged_mask = nm.notna().values & (raw_model != nm_str)
    masks = {"all": np.ones(len(y_te), dtype=bool), "merged": merged_mask}

    print(f"[rs={rs}]  merged rows: {merged_mask.sum():>4} / {len(y_te):,}")
    for mk in MODEL_KINDS:
        yp_r = fit_predict(mk, df_tr_r, df_te_r, RAW_NUM, RAW_CAT, config_raw)
        yp_c = fit_predict(mk, df_tr_c, df_te_c, COMBINED_NUM, COMBINED_CAT, config_combined)

        ape_r = np.abs((yp_r - y_te) / y_te)
        ape_c = np.abs((yp_c - y_te) / y_te)
        metrics[(mk, "raw")]["mape"].append(np.mean(ape_r))
        metrics[(mk, "raw")]["mdape"].append(np.median(ape_r))
        metrics[(mk, "raw")]["mae"].append(np.mean(np.abs(yp_r - y_te)))
        metrics[(mk, "norm")]["mape"].append(np.mean(ape_c))
        metrics[(mk, "norm")]["mdape"].append(np.median(ape_c))
        metrics[(mk, "norm")]["mae"].append(np.mean(np.abs(yp_c - y_te)))

        for s, mask in masks.items():
            for cfg, yp in [("raw", yp_r), ("norm", yp_c)]:
                mp, md, mae, n = slice_stats(y_te, yp, mask)
                slice_metrics[(mk, cfg, s)]["mape"].append(mp)
                slice_metrics[(mk, cfg, s)]["mdape"].append(md)
                slice_metrics[(mk, cfg, s)]["mae"].append(mae)
                slice_metrics[(mk, cfg, s)]["n"].append(n)

        print(
            f"        {mk:<8}  all: raw={ape_r.mean():.4f}  norm={ape_c.mean():.4f}  "
            f"merged: raw={slice_metrics[(mk,'raw','merged')]['mape'][-1]:.4f}  "
            f"norm={slice_metrics[(mk,'norm','merged')]['mape'][-1]:.4f}"
        )

[rs=0]  merged rows: 3236 / 15,147
        catboost  all: raw=0.1955  norm=0.1890  merged: raw=0.1878  norm=0.1812
        xgboost   all: raw=0.2031  norm=0.1905  merged: raw=0.1967  norm=0.1902


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but L

        lightgbm  all: raw=0.1980  norm=0.1911  merged: raw=0.1952  norm=0.1872
[rs=1]  merged rows: 3262 / 15,147
        catboost  all: raw=0.1953  norm=0.1905  merged: raw=0.1903  norm=0.1801
        xgboost   all: raw=0.1975  norm=0.1919  merged: raw=0.1898  norm=0.1840


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but L

        lightgbm  all: raw=0.1970  norm=0.1902  merged: raw=0.1925  norm=0.1829
[rs=2]  merged rows: 3242 / 15,147
        catboost  all: raw=0.1910  norm=0.1860  merged: raw=0.2016  norm=0.1907
        xgboost   all: raw=0.1938  norm=0.1855  merged: raw=0.2088  norm=0.1920


...

/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but L

        lightgbm  all: raw=0.1881  norm=0.1826  merged: raw=0.1848  norm=0.1797
[rs=9]  merged rows: 3335 / 15,147
        catboost  all: raw=0.2046  norm=0.1983  merged: raw=0.1800  norm=0.1719
        xgboost   all: raw=0.2015  norm=0.1964  merged: raw=0.1870  norm=0.1802


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


        lightgbm  all: raw=0.2045  norm=0.1972  merged: raw=0.1880  norm=0.1798


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [8]:
def fmt_delta(d, base, is_count=False):
    rel = d / base * 100 if base > 0 else float("nan")
    if is_count:
        return f"{d:+,.0f} ({rel:+.1f}%)"
    return f"{d:+.4f} ({rel:+.1f}%)"


# ── Headline summary (full test set) ─────────────────────────────────────
print(f"\n{'='*92}")
print(f"  Headline (10-seed average, full test set)")
print(f"{'='*92}")
print(f"  {'Model':<10} {'Config':<10} {'Avg MAPE':>10}  {'Avg MdAPE':>10}  {'Avg MAE':>14}")
print(f"  {'-'*88}")
for mk in MODEL_KINDS:
    raw_mp  = np.mean(metrics[(mk, 'raw')]['mape'])
    raw_md  = np.mean(metrics[(mk, 'raw')]['mdape'])
    raw_mae = np.mean(metrics[(mk, 'raw')]['mae'])
    norm_mp  = np.mean(metrics[(mk, 'norm')]['mape'])
    norm_md  = np.mean(metrics[(mk, 'norm')]['mdape'])
    norm_mae = np.mean(metrics[(mk, 'norm')]['mae'])
    print(f"  {mk:<10} {'raw':<10} {raw_mp:>10.4f}  {raw_md:>10.4f}  {raw_mae:>14,.0f}")
    print(f"  {mk:<10} {'norm':<10} {norm_mp:>10.4f}  {norm_md:>10.4f}  {norm_mae:>14,.0f}")
    print(f"  {'':<10} {'  Δ':<10} {fmt_delta(norm_mp - raw_mp, raw_mp):>10}  "
          f"{fmt_delta(norm_md - raw_md, raw_md):>10}  "
          f"{fmt_delta(norm_mae - raw_mae, raw_mae, is_count=True):>14}")
    print()

# ── Per-slice breakdown ──────────────────────────────────────────────────
print(f"\n{'='*100}")
print(f"  Per-slice performance (long format, 10-seed average)")
print(f"{'='*100}")
print(f"  {'Model':<10} {'Slice':<10} {'Config':<6} {'Avg n':>9}  {'Avg MAPE':>17}  {'Avg MdAPE':>17}  {'Avg MAE':>21}")
print(f"  {'-'*100}")
for s in SLICES:
    for mk in MODEL_KINDS:
        for cfg in ["raw", "norm"]:
            m = slice_metrics[(mk, cfg, s)]
            avg_n = np.mean(m["n"])
            mp = np.nanmean(m['mape']); md = np.nanmean(m['mdape']); mae = np.nanmean(m['mae'])
            print(f"  {mk:<10} {s:<10} {cfg:<6} {avg_n:>9,.0f}  {mp:>17.4f}  {md:>17.4f}  {mae:>21,.0f}")
        raw_mp   = np.nanmean(slice_metrics[(mk, 'raw',  s)]['mape'])
        norm_mp  = np.nanmean(slice_metrics[(mk, 'norm', s)]['mape'])
        raw_md   = np.nanmean(slice_metrics[(mk, 'raw',  s)]['mdape'])
        norm_md  = np.nanmean(slice_metrics[(mk, 'norm', s)]['mdape'])
        raw_mae  = np.nanmean(slice_metrics[(mk, 'raw',  s)]['mae'])
        norm_mae = np.nanmean(slice_metrics[(mk, 'norm', s)]['mae'])
        cell_mp  = fmt_delta(norm_mp - raw_mp, raw_mp)
        cell_md  = fmt_delta(norm_md - raw_md, raw_md)
        cell_mae = fmt_delta(norm_mae - raw_mae, raw_mae, is_count=True)
        print(f"  {'':<10} {'':<10} {'  Δ':<6} {'':>9}  {cell_mp:>17}  {cell_md:>17}  {cell_mae:>21}")
        print()
    print(f"  {'-'*100}")

print(f"\n  Notes:")
print(f"    * 'merged' = rows where raw model string was canonicalized to a different value")
print(f"                  (synonym redirects like 'golf 7' -> 'golf', '220' -> 'c class').")
print(f"    * Smaller MAPE / MdAPE / MAE = better; negative Δ = normalization wins.")


  Headline (10-seed average, full test set)
  Model      Config       Avg MAPE   Avg MdAPE         Avg MAE
  ----------------------------------------------------------------------------------------
  catboost   raw            0.1972      0.0927          26,676
  catboost   norm           0.1920      0.0893          25,260
               Δ        -0.0052 (-2.7%)  -0.0034 (-3.6%)  -1,416 (-5.3%)

  xgboost    raw            0.1990      0.1025          27,289
  xgboost    norm           0.1908      0.0962          25,964
               Δ        -0.0082 (-4.1%)  -0.0063 (-6.2%)  -1,325 (-4.9%)

  lightgbm   raw            0.1991      0.1054          27,625
  lightgbm   norm           0.1918      0.0993          26,419
               Δ        -0.0074 (-3.7%)  -0.0061 (-5.8%)  -1,205 (-4.4%)


  Per-slice performance (long format, 10-seed average)
  Model      Slice      Config     Avg n           Avg MAPE          Avg MdAPE                Avg MAE
  -----------------------------------------